# Waste Image Classification

A deep learning project for classifying waste images into six categories using transfer learning with MobileNetV2.

## Project workflow

1. Download and prepare the dataset
2. Preprocess and split the images
3. Apply data augmentation
4. Build a MobileNetV2 transfer-learning model
5. Train and evaluate the classifier
6. Test single-image predictions
7. Demonstrate the model with a Gradio interface

**Development environment:** Google Colab was used during development and testing.


In [2]:
# Install and import libraries
!pip install -q opendatasets gradio

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import opendatasets as od

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
# Download dataset from Kaggle
od.download("https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification")

# Google Drive or local folder structure.
dataset_path = os.path.join(
    "garbage-classification",
    "Garbage classification",
    "Garbage classification"
)

print(os.listdir(dataset_path))


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds


In [ ]:
# Load and preprocess images
image_size = 224
data, labels = [], []

class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
class_map = {name: idx for idx, name in enumerate(class_names)}

for class_name in class_names:
    class_path = os.path.join(dataset_path, class_name)

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)

        if img is not None:
            img = cv2.resize(img, (image_size, image_size))
            img = img / 255.0
            data.append(img)
            labels.append(class_map[class_name])

data = np.array(data)
labels = np.array(labels)

print("Images:", len(data))
print("Classes:", class_names)


In [ ]:
# Split dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(
    data, labels, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples:       {len(X_test)}")


In [ ]:
# Data augmentation
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_datagen.fit(X_train)


In [ ]:
# Build MobileNetV2 transfer-learning model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Input, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

base_model = MobileNetV2(
    input_shape=(image_size, image_size, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the pretrained feature extractor
base_model.trainable = False

# Add a custom classification head
inputs = Input(shape=(image_size, image_size, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
# Train the model
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)


In [ ]:
# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)

print(f"Test accuracy: {test_acc * 100:.2f}%")


In [ ]:
# Predict a single image
arabic_labels = {
    'cardboard': 'كرتون',
    'glass': 'زجاج',
    'metal': 'معدن',
    'paper': 'ورق',
    'plastic': 'بلاستيك',
    'trash': 'نفايات'
}

def predict_image(img_path):
    img = cv2.imread(img_path)

    if img is None:
        print("Could not load the image.")
        return

    img = cv2.resize(img, (image_size, image_size))
    img = np.expand_dims(img, axis=0) / 255.0

    prediction = model.predict(img, verbose=0)
    class_idx = np.argmax(prediction)
    class_name = class_names[class_idx]

    print(f"Predicted class: {class_name}")
    print(f"Arabic label: {arabic_labels[class_name]}")

# Example using an image from the downloaded dataset
example_image = os.path.join(dataset_path, "glass", "glass1.jpg")
predict_image(example_image)


In [ ]:
# Gradio prediction interface
import gradio as gr

def classify_image_text_only(image):
    img = cv2.resize(image, (image_size, image_size))
    img = np.expand_dims(img, axis=0) / 255.0

    prediction = model.predict(img, verbose=0)
    class_idx = np.argmax(prediction)
    class_name = class_names[class_idx]

    return f"Predicted category: {class_name} ({arabic_labels[class_name]})"

interface_text = gr.Interface(
    fn=classify_image_text_only,
    inputs=gr.Image(type="numpy", label="Upload a waste image"),
    outputs=gr.Textbox(label="Prediction"),
    title="Waste Image Classification",
    description="Upload an image to predict its waste category."
)

# Use share=True only when a temporary public Gradio link is needed.
interface_text.launch(share=False)


## Notes

- The notebook was developed and tested using Google Colab.
- The project uses MobileNetV2 transfer learning for six-class waste image classification.
- A Gradio interface was used as an interactive demonstration during development.
- The trained model achieved **93.42% test accuracy** in the original project run.
